In [1]:
import os
import torch
import wandb
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader, random_split


In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ATRI SUKUL\_netrc.
wandb: Currently logged in as: atrisukul1508 (atrisukul1508-iittp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [41]:
wandb.init(project="alexnet-project-pytorch",config={
    "epochs":10,
    "batch_size":16,
    "learning_rate":0.001,
    "architecture":"alexnet",
    "pretrained":True,
    "input_size":128
})

In [42]:
config = wandb.config

In [5]:
transform = transforms.Compose([
    transforms.Resize((config.input_size,config.input_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.405],[0.229,0.224,0.225])
])
train_dir = './flowers/train'
val_dir = './flowers/val'

train_dataset = datasets.ImageFolder(root=train_dir,transform=transform)
val_dataset = datasets.ImageFolder(root=val_dir,transform=transform)

In [43]:
train_dl = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_dl = DataLoader(val_dataset, batch_sampler=config.batch_size)

In [44]:
alexnet = models.alexnet(pretrained= config.pretrained)

for param in alexnet.parameters():
    param.requires_grad = False 

alexnet.classifier[6] = nn.Sequential(
    nn.Linear(4096,128),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(128,5)
)

for param in alexnet.classifier[6].parameters():
    param.requires_grad_(True)

device = torch.device(['cpu','cuda'][torch.cuda.is_available()])

alexnet = alexnet.to(device)

c:\Users\ATRI SUKUL\deeplearning\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ATRI SUKUL\deeplearning\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet.parameters(), lr=config.learning_rate)

In [45]:
wandb.watch(alexnet, criterion, log='all', log_freq=50, log_graph=True)

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


In [46]:
def train_batch(model, criterion, optimizer, batch_img, batch_labels):
    output = model(batch_img)
    # print(output.shape, batch_labels)
    loss = criterion(output, batch_labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss

In [47]:
def train_log(loss, epoch, example_seen):
    wandb.log({'loss':loss, 'epoch':epoch}, step=example_seen)

    if (epoch+1)%100 == 0:
        print(f"Loss: {loss.item()}, epoch: {epoch}, example seen: {example_seen}")

In [51]:
def val_model(model, val_dl, epoch):
    model.eval()
    val_total = 0
    val_correct = 0
    for image, label in val_dl:
        output = model(image)
        _, preds = torch.max(output,1)
        val_correct += (preds == label).sum().item()
        val_total += len(label)

    val_acc = val_correct/val_total
    wandb.log({"epoch":epoch,"val_acc":val_acc})


In [ ]:
def train_model(model, criterion, optimizer, train_dl, val_dl, config):

    example_seen = 0
    batch_seen = 0

    for epoch in range(config.epochs):
        model.train()
        for i, (images, labels) in enumerate(train_dl):

            loss = train_batch(model, criterion, optimizer, images, labels)

            example_seen += images.shape[0]
            batch_seen += 1

            if ((batch_seen+1)%25 ==0 ):
                train_log(loss, epoch, example_seen)

        val_model(model, val_dl, epoch)


In [ ]:
def model_pipeline(hyperparameters):
    model = alexnet

    train_model(alexnet, criterion, optimizer, train_dl, hyperparameters)

    return model   

In [50]:
model = model_pipeline(config)